# VFIformer — Video Frame Interpolation Pipeline

## What this notebook demonstrates

This notebook doubles the frame rate of a video using **VFIformer** (Video Frame Interpolation Transformer), a neural network that synthesises new intermediate frames between existing ones. It is **Step 6** in the Rekrea video enhancement workflow and is designed to run between the two passes of the Real-ESRGAN notebook.

### How frame interpolation works

Standard video runs at 24–60 frames per second (fps). The human visual system perceives motion as fluid when the frame rate is high enough. Interpolation inserts *synthetic* intermediate frames to smooth motion:

```
Original:         frame_1 ──────────────────── frame_2
After 2× interp:  frame_1 ── frame_1.5 (new) ── frame_2
```

A naive approach would blend the two frames pixel-by-pixel, but this produces **ghosting**: objects appear transparent mid-motion because their pixels overlap from two positions. VFIformer instead uses **optical flow estimation** — it models where each pixel moved between the two frames and synthesises a correctly positioned intermediate frame.

### How VFIformer works

VFIformer is a **Transformer-based** model (Lu et al., 2022). Unlike CNN-based interpolation methods that apply convolutions locally, transformers use self-attention to capture long-range dependencies — critical for large motions that span many pixels.

The model:
1. Estimates a **bidirectional optical flow** between the two input frames (how each pixel moves forward and backward in time).
2. Uses the estimated flow to **warp** both frames towards the midpoint in time.
3. A **transformer decoder** fuses the warped frames and inpaints occluded regions — areas that were hidden in one frame but visible in the other.

### Where this fits in the Rekrea workflow

```
Real-ESRGAN notebook (Pass 1):   Extract → Restore → Downscale → Save to Drive
                                                                        ↓
This notebook (VFIformer):        Load → Resize → Interpolate → Merge → Save to Drive
                                                                        ↓
Real-ESRGAN notebook (Pass 2):   Load → (Optional) Superscale → Reassemble → Audio
```

### Why interpolation is a separate notebook

VFIformer and Real-ESRGAN both require significant GPU memory. Loading both models in the same Colab session would exceed VRAM limits. The Drive handoff pattern lets each model run in its own clean session without competing for memory.

**Steps in this notebook:**
1. Mount Google Drive
2. Clone the VFIformer repository
3. Install dependencies
4. Prepare folder structure and Drive paths
5. Download pretrained model checkpoint (5a: auto or 5b: from Drive)
6. Load restored frames from Drive (output of Real-ESRGAN notebook, Step 5.2b)
7. Patch VFIformer's `demo.py` for PyTorch compatibility
8. Run interpolation via the batch wrapper auxiliary script
9. Merge original and interpolated frames into a unified sequence
10. Save merged frames to Drive for Real-ESRGAN notebook Pass 2

## Step 1 — Mount Google Drive

Drive must be mounted at the start because this notebook:
- **Reads** the downscaled restored frames archive saved by the Real-ESRGAN notebook (`VideoRestoration/downscaled/`).
- **Writes** the merged interpolated frames archive back to Drive for the second ESRGAN pass (`VideoInterpolation/VFI/output/`).
- **Caches** the pretrained model checkpoint to avoid re-downloading it each session.

> **Prerequisite:** The Real-ESRGAN notebook must have completed Steps 5.2a (downscale) and 5.2b (save to Drive) before this notebook can run. If the archive `restored_frames_fps_*_ds.tar.gz` does not exist in Drive, go back and complete those steps first.

In [ ]:
# 📂 Step 1: Mount Google Drive to access enhanced frames
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


## Step 2 — Clone the VFIformer repository

VFIformer is not distributed as a pip package. Installing it means cloning the GitHub repository directly and using its scripts as the inference entry point.

The key script is `demo.py` — it takes two frame image paths (`--img0_path`, `--img1_path`) and a model checkpoint (`--resume`) and writes a single interpolated frame to a specified folder (`--save_folder`). It is designed for single-pair use; the batch wrapper in Step 8 handles iterating over all pairs.

`%cd VFIformer` changes the notebook's working directory to the cloned repository. All subsequent relative paths (e.g. `demo.py`, `pretrained_models/`) resolve from there.

> **Repository:** https://github.com/dvlab-research/VFIformer

In [ ]:
# 💾 Step 2: Clone the repo
!git clone https://github.com/dvlab-research/VFIformer
%cd VFIformer

## Step 3 — Install dependencies

| Package | Role |
|---|---|
| `imageio` | Image file I/O — used by VFIformer internally to read and write frames. |
| `scikit-image` | Image processing utilities (e.g. structural similarity metrics) used internally by VFIformer. |
| `einops` | Tensor rearrangement operations (`rearrange`, `repeat`) used extensively in transformer implementations to reshape feature maps between attention heads. |
| `tqdm` | Progress bars for training loops (present in the codebase even when only running inference). |
| `tensorboardX` | Logging infrastructure required by VFIformer's training code — imported even during inference. |

PyTorch and torchvision are pre-installed in Colab and do not need to be installed here.

In [ ]:
# ⚙️ Step 3: Install minimal extra requirements for Colab
%pip install imageio scikit-image einops tqdm tensorboardX

## Step 4 — Prepare folder structure

This cell declares all paths used in the notebook for both the local Colab runtime and the Drive environment.

| Variable | Path | Role |
|---|---|---|
| `VFI_MODEL` | `net_220.pth` | Pretrained checkpoint filename |
| `LOCAL_DIR` | `/content` | Root of the Colab runtime filesystem |
| `ENV_DIR` | `MyDrive/Rekrea` | Rekrea's persistent root in Google Drive |
| `SCRIPT_DIR` | `/content/scripts` | Local directory for auxiliary scripts |
| `PROJ_MODEL_DIR` | `/content/VFIformer/pretrained_models` | Where VFIformer expects its checkpoint to live |
| `DRV_MODEL_DIR` | `MyDrive/Rekrea/models/VFIformer/pretrained_models` | Persistent model storage in Drive |

Storing the model checkpoint in Drive means it only needs to be downloaded once. Subsequent sessions copy it from Drive to the runtime in seconds instead of re-downloading 230 MB.

In [ ]:
# 📂 Step 4: Folder preparation (runtime and Google Drive environments)
from pathlib import Path

VFI_MODEL = "net_220.pth"
LOCAL_DIR = Path("/content")
ENV_DIR = Path("/content/drive/MyDrive/Rekrea")
SCRIPT_DIR = LOCAL_DIR / "scripts"
PROJ_MODEL_DIR = LOCAL_DIR / "VFIformer/pretrained_models"
DRV_MODEL_DIR = ENV_DIR / "models/VFIformer/pretrained_models"

# Check if directory for auxiliar script exists
if not SCRIPT_DIR.exists():
    SCRIPT_DIR.mkdir(parents=True)
    print("Script directory", SCRIPT_DIR, "created in runtime")

# Check if Drive directory for pretrained model exists
if not DRV_MODEL_DIR.exists():
    DRV_MODEL_DIR.mkdir(parents=True)
    print("Model directory", DRV_MODEL_DIR, "created in Google Drive")

## Step 5 — Download the pretrained model

The pretrained VFIformer checkpoint (`net_220.pth`, ~230 MB, trained for 220 epochs on the Vimeo-90K dataset) must be placed in `VFIformer/pretrained_models/` before inference can run. Two options are provided:

```
Is net_220.pth already saved in Drive (Rekrea/models/VFIformer/pretrained_models/)?
  ├─ No  → Run Step 5a (auto download from HuggingFace — slow, no intervention needed)
  └─ Yes → Run Step 5b (copy from Drive — fast, seconds instead of minutes)
```

Run **either 5a or 5b** — not both. Once downloaded, the model is stored in Drive so all future sessions use the faster Step 5b path.

> **Model source:** `net_220.pth` is hosted on HuggingFace at
> https://huggingface.co/xmanifold/vfiformer/tree/main/pretrained_VFIformer

### Step 5a — Auto download from HuggingFace (slow, no intervention required)

`wget` downloads `net_220.pth` directly from HuggingFace (~230 MB) and moves it into `VFIformer/pretrained_models/`. This works on any fresh session without manual steps but consumes network bandwidth every time.

The cell checks whether the model already exists in `PROJ_MODEL_DIR` before downloading, so it is safe to re-run.

In [ ]:
# 🧠 Step 5a: Pretrained checkpoint download
from pathlib import Path
import shutil

MODEL_PATH = PROJ_MODEL_DIR / VFI_MODEL

# Check if pretrained model exists otherwise download it
if not MODEL_PATH.is_file():
    print("Pretrained model", VFI_MODEL, 
          "not found in directory:", PROJ_MODEL_DIR, 
          "\nDownloading from Huggingface.")

    # Download VFI pretrained model
    # Corrected URL to download the raw file from Hugging Face
    !wget -O /content/net_220.pth "https://huggingface.co/xmanifold/vfiformer/resolve/main/pretrained_VFIformer/net_220.pth"
    print("Pretrained model", VFI_MODEL, 
          "downloaded to local environment.", 
          "\nMoving to Rekrea environment.")

    src = Path("/content/", VFI_MODEL)

    # Move pretrained model to Rekrea environment
    shutil.move(str(src), str(MODEL_PATH))
    print("Pretrained model", VFI_MODEL, "saved to:", MODEL_PATH)

else:
    print(VFI_MODEL, "found in directory:", PROJ_MODEL_DIR)

### Step 5b — Copy from Drive (fast, requires prior download)

If `net_220.pth` is already saved to Drive from a previous session, this cell copies it to the local runtime in seconds — no internet download needed.

**If this is your first time and neither 5a nor 5b has run before:**
1. Download `net_220.pth` from:
   https://huggingface.co/xmanifold/vfiformer/blob/main/pretrained_VFIformer/net_220.pth
2. Upload it to Drive at: `MyDrive/Rekrea/models/VFIformer/pretrained_models/`
3. Run this cell.

In [ ]:
# 🧠 Step 5b: Pretrained checkpoint download
from pathlib import Path
import shutil

LOCAL_MODEL_PATH = PROJ_MODEL_DIR / VFI_MODEL
DRV_MODEL_PATH = DRV_MODEL_DIR / VFI_MODEL

# Check if pretrained model exists otherwise request to download it
if not DRV_MODEL_PATH.is_file():
    print("Pretrained model", VFI_MODEL, 
          "not found in directory:", DRV_MODEL_DIR, 
          "\nFollow the download instructions and re-run the cell.")
else:
    print(VFI_MODEL, "found in directory:", DRV_MODEL_DIR, 
          "\nCopying from Drive to", PROJ_MODEL_DIR)

    # Copy pretrained model from Drive to local environment
    shutil.copy2(str(DRV_MODEL_PATH), str(LOCAL_MODEL_PATH))
    print("Pretrained model", VFI_MODEL, "saved to:", LOCAL_MODEL_PATH)

## Step 6 — Load restored frames from Drive

This cell loads the **downscaled restored frames** that were saved by the Real-ESRGAN notebook at Step 5.2b. It searches the `VideoRestoration/downscaled/` Drive directory for the most recently modified archive matching `restored_frames_fps_*_ds.tar.gz`.

The `_ds` suffix distinguishes downscaled frames (intended for interpolation) from full-resolution restored frames. The FPS value embedded in the filename (e.g. `fps_30_ds`) is extracted and stored in `cfr` — this value is used in Step 10 to name the output archive so the Real-ESRGAN notebook can recover the frame rate without re-probing the original video.

> If no matching archive is found, return to the Real-ESRGAN notebook and complete **Step 5.2a** (downscale restored frames) and **Step 5.2b** (save to Drive) before continuing here.

In [ ]:
# 📁 Step 6 Upload restored frames (downscaled)
from pathlib import Path
import re

DRV_RES_DIR = ENV_DIR / "VideoRestoration/downscaled"
LOCAL_IN_DIR  = LOCAL_DIR / "downscaled_frames"
LOCAL_IN_DIR.mkdir(parents=True, exist_ok=True)

# Upload restored frames
if DRV_RES_DIR.is_dir():
  pattern = "restored_frames_fps_*_ds.tar.gz"
  candidates = sorted(
      DRV_RES_DIR.glob(pattern),
      key=lambda p: p.stat().st_mtime,
      reverse=True
  )

  TAR_PATH = candidates[0]  # newest match
  print("Using:", TAR_PATH)

  # Extract FPS back from filename
  match = re.search(r"fps_(\d+)_ds\.tar\.gz$", TAR_PATH.name)
  
  if match:
      cfr = int(match.group(1))
      print("Recovered FPS from filename:", cfr)

  # -x extract, -z gzip, -f file, -C output directory
  !tar -xzf "{TAR_PATH}" -C "{LOCAL_IN_DIR}"

  print("Done. Extracted into:", LOCAL_IN_DIR)

  if not candidates:
     raise FileNotFoundError("No matching restored_frames archives found.")

## Step 7 — Patch VFIformer's `demo.py` for PyTorch compatibility

VFIformer's `demo.py` loads its checkpoint with:
```python
load_net = torch.load(load_path, map_location=torch.device('cpu'))
```

In PyTorch ≥ 2.0, `torch.load` requires `weights_only` to be set explicitly. The safe default changed from `False` to `True` as a security measure — `weights_only=True` restricts what the unpickler can reconstruct. VFIformer's checkpoint contains non-tensor objects, so `True` breaks loading.

This cell patches the line in-place with `sed`, adding `weights_only=False` to restore the original behaviour explicitly:
```python
load_net = torch.load(load_path, map_location=torch.device('cpu'), weights_only=False)
```

> This is the same category of compatibility issue as the `torchvision.transforms.functional_tensor` patch in the Real-ESRGAN notebook — a repository frozen at a point in time running against a moving PyTorch ecosystem. The local `rekrea` module handles the equivalent fix with a runtime shim instead of a file patch.

In [ ]:
# 🩹 Step 7: Patch torchvision in the demo.py file 
import subprocess

demo_file = "/content/VFIformer/demo.py"

# Read the content of the file
with open(demo_file, 'r') as f:
    content = f.read()

# Replace the line using sed
# The sed command replaces 'torch.load(load_path, map_location=torch.device('cpu'))' with 'torch.load(load_path, map_location=torch.device('cpu'), weights_only=False)'
# Using 'sed -i' for in-place editing
# Using a different delimiter like '#' because the search string contains '/' and '('
command = [
    "sed",
    "-i",
    "s#load_net = torch.load(load_path, map_location=torch.device('cpu'))#load_net = torch.load(load_path, map_location=torch.device('cpu'), weights_only=False)#g",
    demo_file
]

try:
    subprocess.run(command, check=True)
    print(f"Successfully patched {demo_file}")
except subprocess.CalledProcessError as e:
    print(f"Error patching {demo_file}: {e}")

## Step 8 — Run interpolation via the batch wrapper

VFIformer's `demo.py` interpolates a **single frame pair** at a time. Processing an entire video requires iterating over all consecutive pairs — that is what `vfiformer_batch_wrapper.py` does.

### What the auxiliary script does

The batch wrapper (`notebooks/auxiliary/vfiformer_batch_wrapper.py`):

1. **Resizes** all input frames to a fixed resolution (`720×1280`) into `resized_frames/`. VFIformer has a fixed expected input spatial size — mismatched dimensions cause shape errors inside the model's attention layers.
2. **Iterates** over all consecutive pairs `(frame_i, frame_{i+1})`.
3. For each pair, calls `demo.py` as a subprocess passing `--img0_path`, `--img1_path`, `--resume` (model path), and `--save_folder`.
4. `demo.py` writes one interpolated frame per pair into a dedicated subfolder.

### Output structure produced by demo.py

```
interpolated_frames/
  interp_00000.png/
    frame_00001_inter.png    ← synthetic frame between frame_00001 and frame_00002
  interp_00001.png/
    frame_00002_inter.png    ← synthetic frame between frame_00002 and frame_00003
  ...
```

The nested folder structure (one subdirectory per pair) is an artefact of how `demo.py` was originally designed for single-pair use. Step 9 flattens this into a contiguous numbered sequence.

> The wrapper script is fetched from the Rekrea GitHub repository if not already present in the runtime (`/content/scripts/`). An internet connection is required on first use.

In [ ]:
# ✅ Step 8: Ready to interpolate:
from pathlib import Path

SCRIPT_NAME = "vfiformer_batch_wrapper.py"
SCRIPT_PATH = SCRIPT_DIR / SCRIPT_NAME

if not SCRIPT_PATH.is_file():
    print("Script", SCRIPT_NAME, "not found in directory:", SCRIPT_DIR, "\nDownloading from Rekrea GitHub repo.")

    # Point to auxiliary script in Rekrea GitHub repo
    RAW_URL = (
    "https://raw.githubusercontent.com/"
    "edniix-inova/rekrea/master/"
    "notebooks/auxiliary/vfiformer_batch_wrapper.py"
    )

    # Get the auxiliary script and save it in the Drive Rekrea environment
    !wget -O {SCRIPT_PATH} {RAW_URL}
    print("Script", SCRIPT_NAME, "saved to:", SCRIPT_PATH)

else:
    print("Script", SCRIPT_NAME, "found in directory:", SCRIPT_DIR)

# Run auxiliary script
!python {SCRIPT_PATH}

## Step 9 — Merge original and interpolated frames

After interpolation we have two separate sets of frames that must be **interleaved** into a single contiguous sequence:

- `resized_frames/` — the resized originals (`frame_00001.png`, `frame_00002.png`, …)
- `interpolated_frames/` — one synthetic frame per consecutive pair, in nested subfolders

The merge produces a sequence where each synthetic frame is inserted between its two originals:

```
Input (N frames):    frame_1   frame_2   frame_3  …  frame_N
                       │     ↗    │     ↗               │
Output (2N-1 frames): frame_1  frame_1.5  frame_2  frame_2.5  …  frame_N
```

For N original frames the merged output always has **2N − 1** frames. The final original frame has no interpolated pair (there is no frame beyond it), so it is appended once at the end.

The `MOVE_FILES` flag controls whether files are moved (faster, destroys inputs) or copied (safer, preserves inputs for debugging). The default is `False` (copy). Set it to `True` once the workflow is validated to avoid doubling disk usage.

In [ ]:
# 📁 Step 9: Merge original rezised frames with the interpolated frames
from pathlib import Path
import re
import shutil

RESIZED_DIR = Path("/content/resized_frames")
INTERP_ROOT = Path("/content/interpolated_frames")
OUT_DIR     = Path("/content/merged_frames")

OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- helpers ----
frame_re = re.compile(r"frame_(\d+)\.png$")

def idx_from_frame_name(p: Path) -> int:
    m = frame_re.search(p.name)
    if not m:
        raise ValueError(f"Unexpected frame filename: {p.name}")
    return int(m.group(1))

# Choose whether to MOVE or COPY into merged folder
# MOVE is faster and uses no extra disk, but destroys inputs.
MOVE_FILES = False

def transfer(src: Path, dst: Path):
    if MOVE_FILES:
        src.rename(dst)         # in-place rename (fast inside /content)
    else:
        shutil.copy2(src, dst)  # safer default

# ---- collect originals ----
orig_files = sorted(RESIZED_DIR.glob("frame_*.png"), key=idx_from_frame_name)
if len(orig_files) < 2:
    raise RuntimeError(f"Need at least 2 original frames in {RESIZED_DIR}")

n_orig = len(orig_files)

# ---- merge sequence ----
out_idx = 0

for i in range(n_orig - 1):
    # 1) original frame_i
    src_orig = orig_files[i]
    print(f"Processing: {src_orig}")
    dst_orig = OUT_DIR / f"frame_{out_idx+1:05d}.png"
    print(f"Writing: {dst_orig}")
    transfer(src_orig, dst_orig)
    out_idx += 1

    # 2) interpolated frame between i and i+1:
    # folder: interp_{i:05d}.png/
    # file:   frame_{i+1:05d}_interp.png
    interp_dir  = INTERP_ROOT / f"interp_{i:05d}.png"
    interp_file = interp_dir / f"frame_{i+1:05d}_inter.png"
    print(f"Processing: {interp_file}")

    if not interp_file.is_file():
        raise FileNotFoundError(f"Missing interpolated frame: {interp_file}")

    dst_interp = OUT_DIR / f"frame_{out_idx+1:05d}.png"
    print(f"Writing: {dst_interp}")
    transfer(interp_file, dst_interp)
    out_idx += 1

# 3) final original frame (last)
src_last = orig_files[-1]
print(f"Processing: {src_orig}")
dst_last = OUT_DIR / f"frame_{out_idx+1:05d}.png"
print(f"Writing: {dst_last}")
transfer(src_last, dst_last)
out_idx += 1

print(f"Merged frames written to: {OUT_DIR}")
print(f"Original frames: {n_orig}")
print(f"Expected merged frames (2N-1): {2*n_orig - 1}")
print(f"Actual merged frames: {out_idx}")

## Step 10 — Compress merged frames and save to Drive

The merged frames are archived into a `.tar.gz` and saved to:
```
MyDrive/Rekrea/VideoInterpolation/VFI/output/merged_frames_fps_N.tar.gz
```

The frame rate embedded in the filename is **doubled** relative to the input. If the restored frames came from a 30 fps source (`cfr=30`), the merged sequence now contains frames for a 60 fps video (`cfr*2 = 60`). The Real-ESRGAN notebook reads this doubled FPS from the filename when assembling the final video, ensuring the output plays at the correct speed.

> After this step, return to the **Real-ESRGAN notebook** and continue at **Step 7** (Load processed frames from Drive). The notebook will automatically select the most recent `merged_frames_fps_*.tar.gz` archive from the VFI output directory.

In [ ]:
# 📁 Step 10: Compress and move output to Google Drive
from pathlib import Path

DRV_INT_DIR = ENV_DIR / "VideoInterpolation/VFI/output"

# Create target directory
if not DRV_INT_DIR.is_dir():
    DRV_INT_DIR.mkdir(parents=True)

# Embedding frame rate in tar name
tar_name = f"merged_frames_fps_{int(cfr*2)}.tar.gz"

# Compress the frames and move to Drive
!tar -czf "{tar_name}" -C /content/merged_frames .
!mv "{tar_name}" "{DRV_INT_DIR}/"

print(f"Saved: {DRV_INT_DIR / tar_name}")

## Reflection — What could be improved?

| Limitation | Possible solution |
|---|---|
| Each frame pair runs as a separate `demo.py` subprocess | Refactor `vfiformer_batch_wrapper.py` to import the VFIformer model directly in Python and call it in a loop — eliminates subprocess overhead and startup cost per pair |
| Fixed input resolution (`720×1280`) in the batch wrapper | Make the target resolution configurable via a command-line argument, or use dynamic padding to handle arbitrary sizes without resizing |
| `weights_only=False` patch must be applied each session | Fork VFIformer and commit the fix upstream, or maintain a patched fork in the Rekrea repository |
| Only 1× interpolation (one intermediate frame per pair) | Run the wrapper a second time using the merged output as input — each pass doubles the frame count again (1× → 3× → 7× frames from original) |
| VFIformer can produce artefacts on large, fast motions | Try RIFE (Real-Time Intermediate Flow Estimation), which is faster and uses a recurrent flow estimation approach better suited to large displacements |
| Nested subfolder output from `demo.py` requires a separate merge step | Modify `demo.py` to write interpolated frames directly into a flat output directory |
| `MOVE_FILES = False` doubles disk usage during merge | Set `MOVE_FILES = True` once the workflow is stable, or stream from interpolation to merge without writing intermediates |

VFIformer's transformer architecture was a significant step forward over earlier CNN-based interpolation methods (e.g. DAIN, CAIN). The gap has since narrowed as RIFE-based models have become faster and higher quality with simpler setup requirements. Understanding both approaches — and the trade-offs between accuracy on large motions, inference speed, and ease of deployment — is the key learning from comparing these notebooks.